In [3]:
import pandas as pd

df = pd.read_csv("../data/kickstarter_us_filtered.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (38504, 17)


,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,237090,GETTING OVER - One son's search to finally kno...,http://www.kickstarter.com/projects/charnick/g...,Film & Video,Documentary,"Los Angeles, CA",successful,6000.0,6535.0,1.089167,100,"Sun, 08 Apr 2012 02:14:00 -0000",13,"$1,$10,$25,$30,$50,$75,$85,$100,$110,$250,$500...",4,0,32.22
3,246101,The Launch of FlyeGrlRoyalty &quot;The New Nam...,http://www.kickstarter.com/projects/flyegrlroy...,Fashion,Fashion,"Novi, MI",failed,3500.0,0.0,0.000000,0,"Wed, 01 Jun 2011 15:25:39 -0000",6,"$10,$25,$50,$100,$150,$250",2,0,30.00
4,316217,Dinner Party - a short film about friendship.....,http://www.kickstarter.com/projects/249354515/...,Film & Video,Short Film,"Portland, OR",successful,3500.0,3582.0,1.023331,39,"Wed, 22 Jun 2011 13:33:00 -0000",7,"$5,$25,$50,$100,$250,$500,$1,000",8,0,21.43


In [4]:
print(df.columns.tolist())

['project id', 'name', 'url', 'category', 'subcategory', 'location', 'status', 'goal', 'pledged', 'funded percentage', 'backers', 'funded date', 'levels', 'reward levels', 'updates', 'comments', 'duration']


In [5]:
categorical_cols = ["category", "subcategory", "location", "status"]

for col in categorical_cols:
    print(f"\n{col.upper()}")
    print("Unique values:", df[col].nunique())
    print("Missing values:", df[col].isna().sum())
    print(df[col].value_counts().head(10))


CATEGORY
Unique values: 14
Missing values: 0
category
Film &amp; Video    10925
Music                9564
Publishing           3779
Art                  3259
Theater              2201
Design               1452
Games                1365
Food                 1243
Photography          1110
Fashion               975
Name: count, dtype: int64

SUBCATEGORY
Unique values: 51
Missing values: 0
subcategory
Short Film          3519
Documentary         3056
Music               2878
Theater             2201
Film &amp; Video    2162
Indie Rock          1733
Rock                1585
Narrative Film      1294
Food                1243
Photography         1110
Name: count, dtype: int64

LOCATION
Unique values: 3772
Missing values: 0
location
Los Angeles, CA      3641
New York, NY         3432
Brooklyn, NY         1523
Chicago, IL          1383
San Francisco, CA    1243
Portland, OR          904
Seattle, WA           857
Austin, TX            759
Boston, MA            723
Nashville, TN         629
Name:

In [6]:
print("Dataset shape:", df.shape)
print("\nStatus distribution:")
print(df["status"].value_counts())

Dataset shape: (38504, 17)

Status distribution:
status
successful    21075
failed        17429
Name: count, dtype: int64


## Prepare Location Features

The original dataset stores city and state together in the `location` column. 
We separate these into individual city and state features so they can be
analyzed and encoded independently.

In [7]:
df[["city", "state"]] = df["location"].str.rsplit(",", n=1, expand=True)

df["city"] = df["city"].str.strip()
df["state"] = df["state"].str.strip()

df[["location", "city", "state"]].head(10)

,location,city,state
0,"Columbia, MO",Columbia,MO
1,"Maplewood, NJ",Maplewood,NJ
2,"Los Angeles, CA",Los Angeles,CA
3,"Novi, MI",Novi,MI
4,"Portland, OR",Portland,OR
5,"Collegedale, TN",Collegedale,TN
6,"Chicago, IL",Chicago,IL
7,"Chicago, IL",Chicago,IL
8,"Chicago, IL",Chicago,IL
9,"Ashland, OR",Ashland,OR


In [8]:
print("Unique cities:", df["city"].nunique())
print("Unique states:", df["state"].nunique())

print("\nMissing cities:", df["city"].isna().sum())
print("Missing states:", df["state"].isna().sum())

print("\nTop states:")
print(df["state"].value_counts().head(15))

Unique cities: 3142
Unique states: 51

Missing cities: 0
Missing states: 0

Top states:
state
CA    8220
NY    6639
TX    1847
IL    1720
FL    1410
WA    1273
MA    1220
PA    1205
OR    1179
GA     965
MI     951
TN     928
NC     751
OH     746
CO     724
Name: count, dtype: int64


## Clean Categorical Variables

Before encoding, categorical features are standardized by removing extra whitespace
and decoding HTML entities. The original categorical columns are retained for EDA
and interpretation.

In [9]:
import html

cols_to_clean = ["category", "subcategory", "city", "state"]

for col in cols_to_clean:
    df[col] = df[col].apply(
        lambda x: html.unescape(x.strip()) if isinstance(x, str) else x
    )

df[cols_to_clean].head()

,category,subcategory,city,state
0,Film & Video,Short Film,Columbia,MO
1,Games,Board & Card Games,Maplewood,NJ
2,Film & Video,Documentary,Los Angeles,CA
3,Fashion,Fashion,Novi,MI
4,Film & Video,Short Film,Portland,OR


In [10]:
for col in cols_to_clean:
    print(f"{col}: {df[col].nunique()} unique values")

print("\nCategories:")
print(df["category"].value_counts())

category: 13 unique values
subcategory: 49 unique values
city: 3142 unique values
state: 51 unique values

Categories:
category
Film & Video    11335
Music            9564
Publishing       3779
Art              3259
Theater          2201
Design           1452
Games            1365
Food             1243
Photography      1110
Fashion           975
Comics            915
Technology        656
Dance             650
Name: count, dtype: int64


## Categorical Variable Preparation

The categorical features used for preprocessing are category, subcategory,
city, and state. Location is separated into city and state before encoding.

Low-cardinality categorical features can be one-hot encoded directly, while
city requires additional consideration because of its high cardinality.

In [11]:
categorical_features = ["category", "subcategory", "city", "state"]

for col in categorical_features:
    print(f"{col}:")
    print(f"  Unique values: {df[col].nunique()}")
    print(f"  Missing values: {df[col].isna().sum()}")

category:
  Unique values: 13
  Missing values: 0
subcategory:
  Unique values: 49
  Missing values: 0
city:
  Unique values: 3142
  Missing values: 0
state:
  Unique values: 51
  Missing values: 0


In [12]:
city_counts = df["city"].value_counts()

print("Total unique cities:", df["city"].nunique())
print("Cities appearing only once:", (city_counts == 1).sum())
print("Cities with fewer than 10 projects:", (city_counts < 10).sum())

print("\nTop 20 cities:")
print(city_counts.head(20))

Total unique cities: 3142
Cities appearing only once: 1475
Cities with fewer than 10 projects: 2807

Top 20 cities:
city
Los Angeles      3641
New York         3434
Brooklyn         1524
Chicago          1383
San Francisco    1243
Portland         1015
Seattle           857
Austin            760
Boston            724
Nashville         631
Philadelphia      555
Atlanta           550
Minneapolis       451
Washington        433
San Diego         379
Denver            346
Detroit           327
New Orleans       303
Orlando           301
Dallas            300
Name: count, dtype: int64


## Handle High-Cardinality City Values

City contains thousands of unique values, many of which occur only a few times.
To reduce dimensionality before encoding, cities with fewer than 10 campaigns
are grouped into an `Other` category.

In [13]:
city_counts = df["city"].value_counts()
frequent_cities = city_counts[city_counts >= 10].index

df["city_grouped"] = df["city"]

df.loc[
    df["city"].notna() & ~df["city"].isin(frequent_cities),
    "city_grouped"
] = "Other"

print("Original unique cities:", df["city"].nunique())
print("Grouped unique cities:", df["city_grouped"].nunique())
print("Missing grouped cities:", df["city_grouped"].isna().sum())

df["city_grouped"].value_counts().head(15)

Original unique cities: 3142
Grouped unique cities: 336
Missing grouped cities: 0


city_grouped
Other            6200
Los Angeles      3641
New York         3434
Brooklyn         1524
Chicago          1383
San Francisco    1243
Portland         1015
Seattle           857
Austin            760
Boston            724
Nashville         631
Philadelphia      555
Atlanta           550
Minneapolis       451
Washington        433
Name: count, dtype: int64

## One-Hot Encode Categorical Features

The prepared categorical variables are converted into numeric features using
one-hot encoding. Unknown categories are ignored so the preprocessing step can
handle new values during future model evaluation.

In [14]:
from sklearn.preprocessing import OneHotEncoder

features_to_encode = [
    "category",
    "subcategory",
    "city_grouped",
    "state"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_data = encoder.fit_transform(df[features_to_encode])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(features_to_encode),
    index=df.index
)

print("Original categorical shape:", df[features_to_encode].shape)
print("Encoded shape:", encoded_df.shape)

encoded_df.head()

Original categorical shape: (38504, 4)
Encoded shape: (38504, 449)


,category_Art,category_Comics,category_Dance,category_Design,category_Fashion,category_Film & Video,category_Food,category_Games,category_Music,category_Photography,...,state_SD,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
print("Number of encoded features:", encoded_df.shape[1])
print("Missing values in encoded data:", encoded_df.isna().sum().sum())

print("\nEncoded feature counts:")
for feature in features_to_encode:
    count = sum(col.startswith(feature + "_") for col in encoded_df.columns)
    print(f"{feature}: {count}")

Number of encoded features: 449
Missing values in encoded data: 0

Encoded feature counts:
category: 13
subcategory: 49
city_grouped: 336
state: 51


## Summary

Categorical variables were prepared for downstream EDA and machine learning.

- `location` was separated into `city` and `state`.
- HTML entities and extra whitespace were cleaned from categorical values.
- `category` and `subcategory` were standardized before encoding.
- Rare cities with fewer than 10 campaigns were grouped into `Other` to reduce high cardinality.
- `category`, `subcategory`, `city_grouped`, and `state` were one-hot encoded using `OneHotEncoder` with unknown-category handling.
- The preprocessing logic is designed to be rerun on the finalized dataset produced by the upstream data preparation tasks.